<a href="https://colab.research.google.com/github/Giangysam/Correlations/blob/main/Diagnosi_Cause_Difetto_RF_SHAP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Caricamento librerie
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.preprocessing import StandardScaler, OneHotEncoder # Removed SimpleImputer from here
from sklearn.impute import SimpleImputer # Corrected import for SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix)
import shap
import matplotlib.pyplot as plt

# CONFIGURAZIONE
CAT_COLS = ['R', 'W', 'V', 'Z', 'AA'] # Explicitly defined categorical columns
STAT_COLS = ['STAT1','STAT2'] # Explicitly defined static/numerical columns (excluding SOGGETTO)
CSV_PATH = 'https://raw.githubusercontent.com/Giangysam/Correlations/main/DATI_RIS_coperti.xlsx'
TARGET_COLUMN = 'TARGET'

# Caricamento dei dati
def carica_dati(path, target_col):
    df = pd.read_excel(path)
    if target_col not in df.columns:
        raise ValueError(f"Colonna target '{target_col}' non trovata. Colonne disponibili: {list(df.columns)}")

    # Drop 'SOGGETTO' column if it exists, as it's typically an ID and not a feature.
    if 'SOGGETTO' in df.columns:
        df = df.drop(columns=['SOGGETTO'])

    X = df.drop(target_col, axis=1)
    y = df[target_col]
    return X, y

# Define N_FOLDS and RANDOM_STATE globally
N_FOLDS = 9
RANDOM_STATE = 42

# Load data immediately after function definition so X and y are available globally
X, y = carica_dati(CSV_PATH, TARGET_COLUMN)

# Helper function to identify column types dynamically
def get_feature_types(X):
    numeric_features = []
    categorical_features = []
    for col in X.columns:
        # Check if the column is explicitly listed in CAT_COLS and force it to be categorical
        if col in CAT_COLS:
            categorical_features.append(col)
        # Otherwise, use pandas dtype inference for numeric
        elif pd.api.types.is_numeric_dtype(X[col]):
            numeric_features.append(col)
        else:
            # If not explicitly categorical and not numeric, assume it's categorical
            categorical_features.append(col)
    return numeric_features, categorical_features


def costruisci_pipeline_rf(colonne_continue, colonne_binarie):

    from sklearn.compose import ColumnTransformer

    preprocessor_rf = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), colonne_binarie),
            ('num', Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler())
            ]), colonne_continue) # Add imputer and scaler for numeric in RF pipeline
        ],
        remainder='drop'
    )

    pipe = Pipeline([
        ("prep", preprocessor_rf),
        ("clf", RandomForestClassifier(
            n_estimators=700,
            max_depth=6,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42, n_jobs=1))
    ])
    return pipe


def costruisci_pipeline_lasso(colonne_continue, colonne_binarie):
    from sklearn.compose import ColumnTransformer

    preprocessor = ColumnTransformer(transformers=[("scale", Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
            ]), colonne_continue),
            ("cat", OneHotEncoder(handle_unknown='ignore', sparse_output=False), colonne_binarie),])
    pipe = Pipeline([("prep", preprocessor),("clf", LogisticRegression(solver="liblinear",
            penalty="l1", # Corrected: Use 'l1' penalty for Lasso regression
            l1_ratio=1,
            C=0.1,
            class_weight="balanced",
            max_iter=5000,
            random_state=RANDOM_STATE))])
    return pipe



def valuta_con_kfold(pipe, X, y, n_folds=N_FOLDS):

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_STATE)
    aucs, f1s, precisions, recalls = [], [], [], []

    for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)
        if hasattr(pipe, 'predict_proba'):
            y_proba = pipe.predict_proba(X_test)[:, 1]
            aucs.append(roc_auc_score(y_test, y_proba))
        else:
            aucs.append(np.nan)

        f1s.append(f1_score(y_test, y_pred))
        precisions.append(precision_score(y_test, y_pred))
        recalls.append(recall_score(y_test, y_pred))

        print(f"Fold {fold_idx}: AUC={aucs[-1]:.3f}  F1={f1s[-1]:.3f}  "
              f"Precision={precisions[-1]:.3f}  Recall={recalls[-1]:.3f}")

    print("\n--- Medie su tutti i fold ---")
    print(f"AUC:       {np.nanmean(aucs):.3f} (+/- {np.nanstd(aucs):.3f})")
    print(f"F1:        {np.mean(f1s):.3f} (+/- {np.std(f1s):.3f})")
    print(f"Precision: {np.mean(precisions):.3f} (+/- {np.std(precisions):.3f})")
    print(f"Recall:    {np.mean(recalls):.3f} (+/- {np.std(recalls):.3f})")

    return {"auc": aucs, "f1": f1s, "precision": precisions, "recall": recalls}


def analisi_shap(pipe, X, y):

    pipe.fit(X, y)

    modello = pipe.named_steps["clf"]
    if "prep" in pipe.named_steps:
        X_transformed = pipe.named_steps["prep"].transform(X)
        feature_names = pipe.named_steps["prep"].get_feature_names_out()
    else:
        X_transformed = X
        feature_names = X.columns

    explainer = shap.TreeExplainer(modello, X_transformed)
    shap_values = explainer.shap_values(X_transformed)

    if isinstance(shap_values, list):
        sv = shap_values[1]
    elif len(shap_values.shape) == 3:
        sv = shap_values[:, :, 1]
    else:
        sv = shap_values

    plt.figure()
    shap.summary_plot(sv, X_transformed, feature_names=feature_names, show=False, max_display=20)
    plt.tight_layout()
    # Changed file path to be relative for Colab compatibility
    plt.savefig("shap_summary_best.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("Salvato: shap_summary.png (le 20 variabili più importanti, con direzione dell'effetto)")

    importanza_media = np.abs(sv).mean(axis=0)
    tabella_importanza = pd.DataFrame({
        "feature": feature_names,
        "importanza_shap_media": importanza_media
    }).sort_values("importanza_shap_media", ascending=False)

    # Changed file path to be relative for Colab compatibility
    tabella_importanza.to_csv("shap_importanza_best.csv", index=False)
    print("Salvato: shap_importanza.csv (ranking completo)")
    print("\nTop 15 variabili per importanza SHAP:")
    print(tabella_importanza.head(15).to_string(index=False))

    return tabella_importanza, sv, feature_names


def analisi_lasso_coefficienti(pipe, X, y):

    pipe.fit(X, y)
    modello = pipe.named_steps["clf"]
    if "prep" in pipe.named_steps:
        feature_names = pipe.named_steps["prep"].get_feature_names_out()
    else:
        feature_names = X.columns

    coef = modello.coef_[0]
    tabella = pd.DataFrame({"feature": feature_names,"coefficiente": coef})
    tabella["abs_coef"] = tabella["coefficiente"].abs()
    tabella = tabella[tabella["abs_coef"] > 1e-6].sort_values("abs_coef", ascending=False)

    # Changed file path to be relative for Colab compatibility
    tabella.to_csv("lasso_coefficienti_best.csv", index=False)
    n_sopravvissute = len(tabella)
    n_totali = len(feature_names)
    print(f"\nLasso: {n_sopravvissute}/{n_totali} feature con coefficiente non-zero")
    print("Salvato: lasso_coefficienti.csv")
    print("\nTop 15 coefficienti (segno positivo = aumenta probabilità difetto):")
    print(tabella.head(15).to_string(index=False))

    return tabella


def confronta_shap_lasso(tabella_shap, tabella_lasso, top_n=20):

    top_shap = set(tabella_shap.head(top_n)["feature"])
    top_lasso = set(tabella_lasso.head(top_n)["feature"])
    intersezione = top_shap & top_lasso

    print(f"\n--- Feature robuste (in top {top_n} sia per SHAP che per Lasso) ---")
    if intersezione:
        for f in sorted(intersezione):
            print(f"  - {f}")
    else:
        print("  Nessuna sovrapposizione diretta: possibile collinearità tra")
        print("  gruppi di variabili, o i due modelli catturano pattern diversi")
        print("  (lineare vs non-lineare). Valutare raggruppamento per dominio fisico.")

    return intersezione


if __name__ == "__main__":
    print("=" * 60)
    print("1. CARICAMENTO DATI")
    print("=" * 60)
    print(f"Righe: {len(X)}, Colonne feature: {X.shape[1]}")
    print(f"Distribuzione target: {y.value_counts().to_dict()}")

    colonne_continue, colonne_binarie = get_feature_types(X)
    print(f"Colonne continue (verranno scalate): {len(colonne_continue)}")
    print(f"Colonne binarie/one-hot (non scalate): {len(colonne_binarie)}")

    # Explicitly cast dtypes for X before pipeline processing
    for col in colonne_continue:
        if col in X.columns:
            X[col] = pd.to_numeric(X[col], errors='coerce')
    for col in colonne_binarie:
        if col in X.columns:
            X[col] = X[col].astype(str)

    print("\n" + "=" * 60)
    print("2. VALUTAZIONE PERFORMANCE - RANDOM FOREST (K-FOLD)")
    print("=" * 60)
    pipe_rf = costruisci_pipeline_rf(colonne_continue, colonne_binarie)
    valuta_con_kfold(pipe_rf, X, y)

    print("\n" + "=" * 60)
    print("3. VALUTAZIONE PERFORMANCE - LASSO (K-FOLD)")
    print("=" * 60)
    pipe_lasso = costruisci_pipeline_lasso(colonne_continue, colonne_binarie)
    valuta_con_kfold(pipe_lasso, X, y)

    print("\n" + "=" * 60)
    print("4. ANALISI DIAGNOSTICA - SHAP (Random Forest su tutti i dati)")
    print("=" * 60)
    pipe_rf_full = costruisci_pipeline_rf(colonne_continue, colonne_binarie)
    tabella_shap, sv, feature_names = analisi_shap(pipe_rf_full, X, y)

    print("\n" + "=" * 60)
    print("5. ANALISI DIAGNOSTICA - LASSO (coefficienti su tutti i dati)")
    print("=" * 60)
    pipe_lasso_full = costruisci_pipeline_lasso(colonne_continue, colonne_binarie)
    tabella_lasso = analisi_lasso_coefficienti(pipe_lasso_full, X, y)

    print("\n" + "=" * 60)
    print("6. CONFRONTO E FEATURE ROBUSTE")
    print("=" * 60)
    confronta_shap_lasso(tabella_shap, tabella_lasso)

    print("\nFatto. File generati: shap_summary.png, shap_importanza.csv, lasso_coefficienti.csv")

1. CARICAMENTO DATI
Righe: 221, Colonne feature: 68
Distribuzione target: {0: 140, 1: 81}
Colonne continue (verranno scalate): 62
Colonne binarie/one-hot (non scalate): 6

2. VALUTAZIONE PERFORMANCE - RANDOM FOREST (K-FOLD)
Fold 1: AUC=0.792  F1=0.625  Precision=0.714  Recall=0.556
Fold 2: AUC=0.910  F1=0.842  Precision=0.800  Recall=0.889
Fold 3: AUC=0.799  F1=0.636  Precision=0.538  Recall=0.778
Fold 4: AUC=0.896  F1=0.778  Precision=0.778  Recall=0.778
Fold 5: AUC=0.833  F1=0.636  Precision=0.538  Recall=0.778
Fold 6: AUC=0.763  F1=0.636  Precision=0.538  Recall=0.778
Fold 7: AUC=0.637  F1=0.375  Precision=0.429  Recall=0.333
Fold 8: AUC=0.793  F1=0.625  Precision=0.714  Recall=0.556
Fold 9: AUC=0.748  F1=0.625  Precision=0.714  Recall=0.556

--- Medie su tutti i fold ---
AUC:       0.797 (+/- 0.077)
F1:        0.642 (+/- 0.121)
Precision: 0.641 (+/- 0.123)
Recall:    0.667 (+/- 0.166)

3. VALUTAZIONE PERFORMANCE - LASSO (K-FOLD)
Fold 1: AUC=0.764  F1=0.609  Precision=0.500  Recall=

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(


Fold 3: AUC=0.806  F1=0.640  Precision=0.500  Recall=0.889
Fold 4: AUC=0.861  F1=0.762  Precision=0.667  Recall=0.889
Fold 5: AUC=0.701  F1=0.667  Precision=0.500  Recall=1.000


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(


Fold 6: AUC=0.733  F1=0.583  Precision=0.467  Recall=0.778
Fold 7: AUC=0.704  F1=0.609  Precision=0.500  Recall=0.778
Fold 8: AUC=0.926  F1=0.824  Precision=0.875  Recall=0.778


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(


Fold 9: AUC=0.733  F1=0.636  Precision=0.538  Recall=0.778

--- Medie su tutti i fold ---
AUC:       0.795 (+/- 0.084)
F1:        0.669 (+/- 0.074)
Precision: 0.564 (+/- 0.122)
Recall:    0.852 (+/- 0.091)

4. ANALISI DIAGNOSTICA - SHAP (Random Forest su tutti i dati)


100%|===================| 440/442 [00:17<00:00]       

Salvato: shap_summary.png (le 20 variabili più importanti, con direzione dell'effetto)
Salvato: shap_importanza.csv (ranking completo)

Top 15 variabili per importanza SHAP:
         feature  importanza_shap_media
         num__AR               0.022983
          num__C               0.020345
         num__AS               0.018747
         num__AH               0.018333
num__TIRO_G4_USC               0.015977
         num__AG               0.014256
          num__B               0.012995
         num__AU               0.010674
         num__AQ               0.009827
         num__BN               0.008144
         num__AX               0.007822
          num__I               0.007042
         num__AI               0.006809
         num__AD               0.006772
         num__BA               0.006688

5. ANALISI DIAGNOSTICA - LASSO (coefficienti su tutti i dati)

Lasso: 11/220 feature con coefficiente non-zero
Salvato: lasso_coefficienti.csv

Top 15 coefficienti (segno positivo = aum

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:1196: UserWarning: l1_ratio parameter is only used when penalty is 'elasticnet'. Got (penalty=l1)
  warnings.warn(
